In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import requests
import numpy as np
import pandas as pd
from pandas import json_normalize
import folium
import time

따릉이 서버에 post 방식으로 요청해서 서버가 응답하는 따릉이 스테이션 데이터를 받는다.

In [2]:
targetSite = 'https://www.bikeseoul.com/app/station/getStationRealtimeStatus.do'
response = requests.post(targetSite, data={
    'stationGrpSeq': 'ALL'
})
# print(response)
# print(type(response.text))
# print(response.text)

텍스트로된 따릉이 스테이션 데이터를 파이썬에서 처리 가능한 자료형으로 변환환다.

In [3]:
bicycle = response.json()
print(type(bicycle))
print(bicycle.keys())

<class 'dict'>
dict_keys(['stationImgPath', 'appUserSessionVO', 'loginYn', 'realtimeList', 'sessionId', 'stationVO', 'checkResult'])


딕셔너리 타입으로 변환환 데이터에서 따릉이 스테이션 정보가 할당된 realtimeList key의 value를 판다스 데이터프레임으로 만든다.

In [4]:
print(len(bicycle['realtimeList']))
print(type(bicycle['realtimeList']))
bicycle['realtimeList'][0]

2750
<class 'list'>


{'stationImgFileName': '',
 'stationId': 'ST-4',
 'stationName': '102. 망원역 1번출구 앞',
 'stationLongitude': '126.91062927',
 'stationLatitude': '37.55564880',
 'rackTotCnt': '15',
 'parkingBikeTotCnt': '0',
 'parkingQRBikeCnt': '3',
 'parkingELECBikeCnt': '0',
 'stationSeCd': 'RAK_002',
 'mode': None}

In [5]:
bicycle_df = json_normalize(bicycle, 'realtimeList')
bicycle_df

,stationImgFileName,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt,stationSeCd,mode
0,,ST-4,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,3,0,RAK_002,None
1,,ST-5,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,2,0,RAK_002,None
2,,ST-6,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,17,1,RAK_002,None
3,,ST-7,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,8,2,RAK_002,None
4,,ST-8,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,9,1,RAK_002,None
...,...,...,...,...,...,...,...,...,...,...,...
2745,,ST-3419,6188.금호아파트,126.86475372,37.55611038,17,0,23,4,RAK_002,None
2746,,ST-3424,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,2,0,RAK_002,None
2747,,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.82738495,37.56614685,10,0,11,2,RAK_002,None
2748,,ST-3437,6191.가양역5번출구,126.85717010,37.56021500,10,0,10,2,RAK_002,None


In [6]:
bicycle_df.columns

Index(['stationImgFileName', 'stationId', 'stationName', 'stationLongitude',
       'stationLatitude', 'rackTotCnt', 'parkingBikeTotCnt',
       'parkingQRBikeCnt', 'parkingELECBikeCnt', 'stationSeCd', 'mode'],
      dtype='str')

bicycle_df 데이터프레임에서 drop() 메소드로 불필요한 컬럼을 제거한다.

stationId: 대여소 ID  
stationName: 대여소 이름  
stationLongitude: 대여소 경도  
stationLatitude: 대여소 위도  
rackTotCnt: 대여소에 주차 가능한 자전거 대수  
parkingBikeTotCnt: 대여소에 주차된 LCD형 따릉이 대수  
parkingQRBikeCnt: 대여소에 주차된 QR형 따릉이 대수  
parkingELECBikeCnt: 대여소에 주차된 새싹 따릉이 대수

In [7]:
bicycle_df[
    ['stationId', 'stationName', 'stationLongitude', 'stationLatitude', 'rackTotCnt', 'parkingBikeTotCnt', 'parkingQRBikeCnt', 'parkingELECBikeCnt']
]

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt
0,ST-4,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,3,0
1,ST-5,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,2,0
2,ST-6,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,17,1
3,ST-7,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,8,2
4,ST-8,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,9,1
...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.86475372,37.55611038,17,0,23,4
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,2,0
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.82738495,37.56614685,10,0,11,2
2748,ST-3437,6191.가양역5번출구,126.85717010,37.56021500,10,0,10,2


In [8]:
# bicycle_df = bicycle_df.drop(columns=['stationImgFileName', 'stationSeCd', 'mode'])
# drop() 메소드에 inplace=True를 지정하면 메소드 실행 결과를 별도로 저장하지 않아도 자동으로 저장된다.
bicycle_df.drop(columns=['stationImgFileName', 'stationSeCd', 'mode'], inplace=True)
bicycle_df

,stationId,stationName,stationLongitude,stationLatitude,rackTotCnt,parkingBikeTotCnt,parkingQRBikeCnt,parkingELECBikeCnt
0,ST-4,102. 망원역 1번출구 앞,126.91062927,37.55564880,15,0,3,0
1,ST-5,103. 망원역 2번출구 앞,126.91083527,37.55495071,14,0,2,0
2,ST-6,104. 합정역 1번출구 앞,126.91508484,37.55073929,13,0,17,1
3,ST-7,105. 합정역 5번출구 앞,126.91482544,37.55000687,5,0,8,2
4,ST-8,106. 합정역 7번출구 앞,126.91282654,37.54864502,12,0,9,1
...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.86475372,37.55611038,17,0,23,4
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.84830475,37.56448364,11,0,2,0
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.82738495,37.56614685,10,0,11,2
2748,ST-3437,6191.가양역5번출구,126.85717010,37.56021500,10,0,10,2


데이터프레임 열이름을 변경하려면 columns 속성에 리스트 형태로 새 열이름을 넣어주면 된다.

In [9]:
bicycle_df.columns = ['id', 'name', 'lon', 'lat', 'totcnt', 'lcd', 'qr', 'elec']

경도, 위도, 자전거 주차 대수의 데이터 타입을 변경한다.

In [10]:
# bicycle_df.dtypes
bicycle_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2750 entries, 0 to 2749
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2750 non-null   str  
 1   name    2750 non-null   str  
 2   lon     2750 non-null   str  
 3   lat     2750 non-null   str  
 4   totcnt  2750 non-null   str  
 5   lcd     2750 non-null   str  
 6   qr      2750 non-null   str  
 7   elec    2750 non-null   str  
dtypes: str(8)
memory usage: 172.0 KB


In [11]:
bicycle_df['lon'] = bicycle_df['lon'].astype(float)
bicycle_df['lat'] = bicycle_df['lat'].astype(float)
bicycle_df['totcnt'] = bicycle_df['totcnt'].astype(int)
bicycle_df['lcd'] = bicycle_df['lcd'].astype(int)
bicycle_df['qr'] = bicycle_df['qr'].astype(int)
bicycle_df['elec'] = bicycle_df['elec'].astype(int)

In [12]:
bicycle_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2750 entries, 0 to 2749
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      2750 non-null   str    
 1   name    2750 non-null   str    
 2   lon     2750 non-null   float64
 3   lat     2750 non-null   float64
 4   totcnt  2750 non-null   int64  
 5   lcd     2750 non-null   int64  
 6   qr      2750 non-null   int64  
 7   elec    2750 non-null   int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 172.0 KB


따릉이 스테이션에 주차된 전체 자전거 대수를 기억하는 파생 변수를 만든다.

In [13]:
bicycle_df['total'] = bicycle_df['lcd'] + bicycle_df['qr'] + bicycle_df['elec']
bicycle_df

,id,name,lon,lat,totcnt,lcd,qr,elec,total
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,3,0,3
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,2,0,2
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,17,1,18
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,8,2,10
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,9,1,10
...,...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,23,4,27
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,11,2,13
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12


In [14]:
bicycle_df.head()

,id,name,lon,lat,totcnt,lcd,qr,elec,total
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,3,0,3
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,2,0,2
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,17,1,18
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,8,2,10
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,9,1,10


In [15]:
bicycle_df.tail()

,id,name,lon,lat,totcnt,lcd,qr,elec,total
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,23,4,27
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,11,2,13
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12
2749,ST-3440,6192.롯데몰김포,126.805962,37.564384,12,0,14,4,18


In [16]:
bicycle_df.to_csv('./data/bicycle.csv', index=False, encoding='utf-8-sig')

In [17]:
df = pd.read_csv('./data/bicycle.csv')
df

,id,name,lon,lat,totcnt,lcd,qr,elec,total
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,3,0,3
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,2,0,2
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,17,1,18
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,8,2,10
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,9,1,10
...,...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,23,4,27
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,11,2,13
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12


서울시 전체 따릉이 스테이션 시각화

In [ ]:
foliumMap = folium.Map(location=[bicycle_df['lat'].mean(), bicycle_df['lon'].mean()], zoom_start=12)
for _, bicycle in bicycle_df.iterrows():
    popup = folium.Popup(f'스테이션 이름: {bicycle["name"].split(".")[1]}<br>주차된 자전거 수: {bicycle["total"]}대', max_width=300)
    folium.Marker(location=[bicycle['lat'], bicycle['lon']], popup=popup).add_to(foliumMap)
foliumMap

# GeoCoding

지오 코딩: 주소로 위도, 경도 정보를 얻어온다.  
역 지오 코딩: 위도, 경도로 주소 정보를 얻어온다.

지오 코딩을 사용하기 위해서 geopy 라이브러리를 설치하고 Nominatim를 import 한다.  
`!pip install geopy`

In [18]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderRateLimited

`GeocoderRateLimited: Non-successful status code 429` 에러 방지를 위해서 Nominatim 객체를 함수가 실행될때마다 생성하지 않고 미리 생성한다.  
`user_agent`에는 다른 사용자와 겹치지 않게 나만의 `고유한 이름`을 설정한다.

In [19]:
# geocoder = Nominatim(user_agent='South Korea', timeout=None)
geocoder = Nominatim(user_agent='my_unique_geocode_id_mkvryu', timeout=None)

## 지오 코딩(주소를 인수로 넘겨받아 위도, 경도를 리턴하는) 함수를 만든다.

In [20]:
def geocoding(address):
    geo = geocoder.geocode(address)
    return {'위도': geo.latitude, '경도': geo.longitude}

address = geocoding('서울시 강남구 역삼동')  
print(address)

address = geocoding('경기도 양주시 양주2동')  
print(address)

## 역 지오 코딩(위도, 경도를 인수로 넘겨받아 주소를 리턴하는) 함수를 만든다.

In [21]:
def geocoding_reverse(lat_lon):
    time.sleep(1.5)
    return geocoder.reverse(lat_lon)

print(geocoding_reverse('37.49988,127.03856'))

address = geocoding_reverse('37.7967589,127.0803125')  
print(type(address))  
print(address)

juso = str(address).split(', ')  
print(type(juso))  
print(juso)  
print(juso[-3], juso[-4], juso[-5])

## 역 지오 코딩 함수를 실행해서 위도와 경도로 얻어지는 구(goo)와 동(dong)을 파생 변수로 추가한다.

In [22]:
bicycle_df['goo'] = np.nan
bicycle_df['dong'] = ''
bicycle_df['goo'] = bicycle_df['goo'].astype(str)
bicycle_df['dong'] = bicycle_df['dong'].astype(str)
bicycle_df

,id,name,lon,lat,totcnt,lcd,qr,elec,total,goo,dong
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,3,0,3,NaN,
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,2,0,2,NaN,
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,17,1,18,NaN,
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,8,2,10,NaN,
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,9,1,10,NaN,
...,...,...,...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,23,4,27,NaN,
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2,NaN,
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,11,2,13,NaN,
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12,NaN,


In [23]:
bicycle_df.dtypes

id            str
name          str
lon       float64
lat       float64
totcnt      int64
lcd         int64
qr          int64
elec        int64
total       int64
goo           str
dong          str
dtype: object

shape 속성으로 데이터프레임에 저장된 전체 데이터의 개수와 각 데이터를 구성하는 열의 개수를 얻어온다.

In [24]:
print(bicycle_df.shape, bicycle_df.shape[0], bicycle_df.shape[1])

(2750, 11) 2750 11


loc, iloc을 사용해서 []에 [행, 열]을 지정해서 특정 행, 열의 데이터를 얻어올 수 있다.  
loc, iloc의 차이점은 loc는 문자열만 사용할 수 있고 iloc는 정수만 사용할 수 있다. 단, 행 인덱스는 제외하고 열 인덱스에만 적용된다.

In [25]:
print(bicycle_df.iloc[0, 2], bicycle_df.loc[0, 'lon'])

126.91062927 126.91062927


## 반복문을 이용해서 각각의 따릉이 스테이션의 위도와 경도를 이용해서 주소를 얻어와 파생 변수에 넣어준다.

In [34]:
'''
for i in range(bicycle_df.shape[0])[:1]:
    # geocoding_reverse() 함수로 전달할 데이터를 만든다.
    lat_lon = f'{bicycle_df.loc[i, "lat"]},{bicycle_df.loc[i, "lon"]}'
    # print(lat_lon)
    # geocoding_reverse() 함수를 실행한다.
    address = geocoding_reverse(lat_lon)
    juso = str(address).split(', ')
    # print(juso[-3], juso[-4], juso[-5])
    
    try:
        bicycle_df.loc[i, 'goo'] = juso[-4]
    except:
        print(f'{i}번째 인덱스의 따릉이 스테이션 -4번째 주소 없음')
        
    try:
        bicycle_df.loc[i, 'dong'] = juso[-5]
    except:
        print(f'{i}번째 인덱스의 따릉이 스테이션 -5번째 주소 없음')
        
    if (i + 1) % 20 == 0:
        print('===============================', i + 1, '===============================')
'''
pass

In [39]:
bicycle_df.to_csv('./data/bicycle_20261109.csv', index=False, encoding='utf-8-sig')

In [42]:
# cp949 어쩌구 저쩌구 에러가 발생되면 encoding='cp949'를 추가한다.
bicycle_df = pd.read_csv('./data/bicycle_20261109.csv')
bicycle_df

,id,name,lon,lat,totcnt,lcd,qr,elec,totla,goo,dong
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,3,0,3,마포구,서교동
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,2,0,2,마포구,망원1동
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,17,1,18,마포구,서교동
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,8,2,10,마포구,서교동
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,9,1,10,마포구,합정동
...,...,...,...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,23,4,27,강서구,염창동
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2,강서구,가양1동
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,11,2,13,강서구,가양1동
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12,강서구,가양3동


## 추가된 파생 변수에 저장된 내용을 확인하고 발생된 문제점을 처리한다.

`isnull()` 메소드와 `isna()` 메소드는 데이터프레임의 특정 열에 저장된 데이터가 NaN이면 True, 그렇치 않으면 False를 리턴한다.

In [47]:
# bicycle_df의 goo열 또는 dong열에 데이터가 채워지지 않은 행을 검색한다.
# '|'는 or 연산자이고 '&'는 and 연산자이다.
bicycle_df[bicycle_df['goo'].isnull() | bicycle_df['dong'].isna()]

,id,name,lon,lat,totcnt,lcd,qr,elec,totla,goo,dong


In [48]:
bicycle_df[bicycle_df['goo'].isnull() | bicycle_df['dong'].isna()]['id']

Series([], Name: id, dtype: str)

In [49]:
errorList = list(bicycle_df[bicycle_df['goo'].isnull() | bicycle_df['dong'].isna()]['id'])
errorList

[]

value_counts() 메소드는 데이터프레임의 특정 열에 저장된 데이터의 종류별 개수를 센다.

In [132]:
bicycle_df['goo'].value_counts()

goo
송파구     217
강서구     195
강남구     174
영등포구    171
노원구     148
서초구     145
강동구     123
마포구     121
구로구     114
양천구     107
종로구      98
은평구      96
중랑구      95
중구       91
성동구      90
동대문구     90
용산구      89
광진구      88
서대문구     82
성북구      80
도봉구      72
금천구      71
동작구      70
관악구      66
강북구      57
Name: count, dtype: int64

In [128]:
# 조건식에 관계연산자를 사용하면 반드시 관계연산자를 사용하는 연산식을 ()로 묶어야 한다. 단, '|' 또는 '&' 연산자를 사용할 경우만 해당된다.
bicycle_df[(bicycle_df['goo'] == '서울특별시') | (bicycle_df['goo'] == '의정부시') | (bicycle_df['goo'] == '하남시')]

,id,name,lon,lat,totcnt,lcd,qr,elec,totla,goo,dong


In [129]:
address = geocoding_reverse('37.484833,127.149918')
address = str(address).split(', ')
print(address)

['북위례2', '657-12', '위례대로', '학암동', '위례동', '송파구', '서울특별시', '경기도', '13011', '대한민국']


In [127]:
bicycle_df.loc[2667, 'goo'] = '송파구'
bicycle_df.loc[2667, 'dong'] = '거여동'

In [130]:
bicycle_df.to_csv('./data/bicycle_20261109.csv', index=False, encoding='utf-8-sig')

In [131]:
bicycle_df = pd.read_csv('./data/bicycle_20261109.csv')
bicycle_df

,id,name,lon,lat,totcnt,lcd,qr,elec,totla,goo,dong
0,ST-4,102. 망원역 1번출구 앞,126.910629,37.555649,15,0,3,0,3,마포구,서교동
1,ST-5,103. 망원역 2번출구 앞,126.910835,37.554951,14,0,2,0,2,마포구,망원1동
2,ST-6,104. 합정역 1번출구 앞,126.915085,37.550739,13,0,17,1,18,마포구,서교동
3,ST-7,105. 합정역 5번출구 앞,126.914825,37.550007,5,0,8,2,10,마포구,서교동
4,ST-8,106. 합정역 7번출구 앞,126.912827,37.548645,12,0,9,1,10,마포구,합정동
...,...,...,...,...,...,...,...,...,...,...,...
2745,ST-3419,6188.금호아파트,126.864754,37.556110,17,0,23,4,27,강서구,염창동
2746,ST-3424,6189.데시앙플렉스 지식산업센터,126.848305,37.564484,11,0,2,0,2,강서구,가양1동
2747,ST-3421,6190.마곡광장(마곡나루역 6번출구),126.827385,37.566147,10,0,11,2,13,강서구,가양1동
2748,ST-3437,6191.가양역5번출구,126.857170,37.560215,10,0,10,2,12,강서구,가양3동


## 따릉이 스테이션 시각화

종로구 따릉이 스테이션 시각화

In [134]:
data = bicycle_df[bicycle_df['goo'] == '종로구']
data.shape

(98, 11)

In [137]:
foliumMap = folium.Map(location=[data['lat'].mean(), data['lon'].mean()], zoom_start=14)
for _, bicycle in data.iterrows():
    popup = folium.Popup(f'스테이션 이름: {bicycle["name"].split(".")[1]}<br>주차된 자전거 수: {bicycle["totla"]}대', max_width=300)
    folium.Marker(location=[bicycle['lat'], bicycle['lon']], popup=popup).add_to(foliumMap)
foliumMap

마포구 서교동 따릉이 스테이션 시각화

In [138]:
data = bicycle_df[(bicycle_df['goo'] == '마포구') & (bicycle_df['dong'] == '서교동')]
data.shape

(12, 11)

In [140]:
foliumMap = folium.Map(location=[data['lat'].mean(), data['lon'].mean()], zoom_start=15)
for _, bicycle in data.iterrows():
    popup = folium.Popup(f'스테이션 이름: {bicycle["name"].split(".")[1]}<br>주차된 자전거 수: {bicycle["totla"]}대', max_width=300)
    folium.Marker(location=[bicycle['lat'], bicycle['lon']], popup=popup).add_to(foliumMap)
foliumMap

따릉이 스테이션 이름에 '마포'가 포함된 따릉이 스테이션 시각화

판다스 데이터프레임의 한 열을 Series라고 하고 Series에는 문자열 메소드인 find()를 사용할 수 없기 때문에 Series를 일시적으로 문자열로 바꾼 후 find() 메소드를 실행해야 하는데 이 때 `str`을 사용한다.

In [146]:
print(type(bicycle_df['name']))
print(type(bicycle_df['name'].str))

<class 'pandas.Series'>
<class 'pandas.core.strings.accessor.StringMethods'>


In [148]:
data = bicycle_df[bicycle_df['name'].str.find('마포') >= 0]
data

,id,name,lon,lat,totcnt,lcd,qr,elec,totla,goo,dong
15,ST-23,121. 마포소방서 앞,126.933174,37.549767,15,0,4,7,11,마포구,신수동
35,ST-204,146. 마포역 2번출구 뒤,126.945824,37.539936,12,0,13,1,14,마포구,용강동
36,ST-205,147. 마포역 4번출구 뒤,126.945915,37.539272,5,0,7,0,7,마포구,도화동
40,ST-211,154. 마포구청역,126.905495,37.560909,11,0,0,2,2,마포구,성산1동
67,ST-343,185. 마포 신수공원 앞,126.934296,37.542545,15,0,2,0,2,마포구,신수동
237,ST-83,407. 마포구 육아종합지원센터,126.883675,37.580631,17,0,18,2,20,마포구,상암동
249,ST-87,419. 홈플러스 마포점 앞,126.899429,37.568420,21,0,4,3,7,마포구,성산2동
251,ST-82,421. 마포구청 앞,126.901863,37.565746,8,0,9,1,10,마포구,성산2동
261,ST-1494,432. 마포중앙도서관,126.908203,37.563339,10,0,19,1,20,마포구,성산1동
1712,ST-2163,3005.마포구청 청사내,126.902184,37.566246,10,0,9,0,9,마포구,성산2동


In [149]:
foliumMap = folium.Map(location=[data['lat'].mean(), data['lon'].mean()], zoom_start=15)
for _, bicycle in data.iterrows():
    popup = folium.Popup(f'스테이션 이름: {bicycle["name"].split(".")[1]}<br>주차된 자전거 수: {bicycle["totla"]}대', max_width=300)
    folium.Marker(location=[bicycle['lat'], bicycle['lon']], popup=popup).add_to(foliumMap)
foliumMap